# Symbolic Layer
The symbolic layer detects simile structures in Arabic sentences using handcrafted rules.  
It identifies patterns like particles (`كأن`, `كأنما`), nouns (`مثل`, `شبيه`), verbs (`يشبه`), and prefixes (`كـ`).  
It outputs structured evidence (`subject`, `particle`, `object`, `confidence`) and provides explanations that complement the neural model's prediction.

In [2]:
from dataclasses import dataclass, field


In [ ]:
SIMILE_PARTICLES_STRONG = {"كأن","كأنما"}
SIMILE_PARTICLES_WEAK = {"كما"}

SIMILE_VERBS = {"يشبه","شبه","يماثل","يضارع"}
SIMILE_NOUNS = {"مثل","شبيه","نظير"}

FALSE_CONTEXT = {"كما أن","كما كان","كما ينبغي"}

def tokenize(text):
    return text.split()

def is_probable_noun(word):
    return word.startswith("ال") or word.endswith("ة") or len(word) > 3

def has_prefix(word):
    return word.startswith("ك") and len(word) > 2


In [ ]:
@dataclass
class SimileStructure:
    subject: str
    particle: str
    object: str
    confidence: float
    rule: str

@dataclass
class Evidence:
    structures: list = field(default_factory=list)
    confidence: float = 0.0
    explanation: list = field(default_factory=list)

In [ ]:
def detect_particle_patterns(words):
    structures = []

    for i in range(1, len(words)-1):
        w = words[i]

        # Strong particles: كأن
        if w in SIMILE_PARTICLES_STRONG:
            left, right = words[i-1], words[i+1]

            if is_probable_noun(left) and is_probable_noun(right):
                structures.append(
                    SimileStructure(
                        subject=left,
                        particle=w,
                        object=right,
                        confidence=0.9,
                        rule="explicit_particle"
                    )
                )

        # Weak particle: كما
        if w in SIMILE_PARTICLES_WEAK:
            left, right = words[i-1], words[i+1]

            if is_probable_noun(left) and is_probable_noun(right):
                structures.append(
                    SimileStructure(
                        subject=left,
                        particle=w,
                        object=right,
                        confidence=0.6,
                        rule="weak_particle"
                    )
                )

    return structures

In [ ]:
def detect_nominal_patterns(words):
    structures = []

    for i in range(1, len(words)-1):
        w = words[i]

        if w in SIMILE_NOUNS:
            left, right = words[i-1], words[i+1]

            if is_probable_noun(left) and is_probable_noun(right):
                structures.append(
                    SimileStructure(
                        subject=left,
                        particle=w,
                        object=right,
                        confidence=0.8,
                        rule="nominal_simile"
                    )
                )

    return structures


def detect_verb_patterns(words):
    structures = []

    for i in range(1, len(words)-1):
        w = words[i]

        if w in SIMILE_VERBS:
            left, right = words[i-1], words[i+1]

            if is_probable_noun(left) and is_probable_noun(right):
                structures.append(
                    SimileStructure(
                        subject=left,
                        particle=w,
                        object=right,
                        confidence=0.75,
                        rule="verb_simile"
                    )
                )

    return structures


def detect_prefix_patterns(words):
    structures = []

    for w in words:
        if has_prefix(w):
            candidate = w[1:]

            if is_probable_noun(candidate):
                structures.append(
                    SimileStructure(
                        subject=None,
                        particle="كـ",
                        object=candidate,
                        confidence=0.55,
                        rule="prefix_simile"
                    )
                )

    return structures


In [ ]:
def has_false_context(sentence):
    for bad in FALSE_CONTEXT:
        if bad in sentence:
            return True
    return False

In [ ]:
def symbolic_detector(sentence):
    words = tokenize(sentence)

    ev = Evidence()

    # Hard rejection if logical context
    if has_false_context(sentence):
        ev.explanation.append("Rejected: logical comparison context")
        ev.confidence = 0.0
        return ev

    structures = []
    structures += detect_particle_patterns(words)
    structures += detect_nominal_patterns(words)
    structures += detect_verb_patterns(words)
    structures += detect_prefix_patterns(words)

    if structures:
        ev.structures = structures
        ev.confidence = max(s.confidence for s in structures)

        for s in structures:
            ev.explanation.append(
                f"[{s.rule}] {s.subject} {s.particle} {s.object}"
            )
    else:
        ev.explanation.append("No simile structure detected")

    return ev

NEURO-SYMBOLIC INTEGRATION

In [ ]:
def neuro_symbolic_classifier(sentence, bert_prob):
    ev = symbolic_detector(sentence)

    if ev.confidence >= 0.85:
        label = 1
        decision = "Symbolic override (clear simile structure)"

    elif bert_prob >= 0.85:
        label = 1
        decision = "Neural confidence high"

    elif bert_prob >= 0.5 and ev.confidence >= 0.5:
        label = 1
        decision = "Neural + symbolic agreement"

    elif bert_prob < 0.4 and ev.confidence < 0.4:
        label = 0
        decision = "Both reject simile"

    elif ev.confidence > bert_prob:
        label = 1
        decision = "Symbolic wins conflict"

    else:
        label = 1 if bert_prob >= 0.5 else 0
        decision = "Fallback to neural"

    explanation = [
        f"BERT probability = {bert_prob:.3f}",
        f"Symbolic confidence = {ev.confidence:.3f}",
        f"Decision = {decision}"
    ] + ev.explanation

    return label, explanation